# IMDB Media Trend Analysis
### A comprehensive Exploratory Data Analysis of global film & television production patterns (1920–2022)

---

## Project Overview

This notebook analyses the **IMDB Movies Dataset** sourced from Kaggle, containing  
approximately **2.59 million titles** spanning over a century of global media production — (1920 - 2022)


| Attribute        | Detail                                      |
|------------------|---------------------------------------------|
| Source           | Kaggle — IMDB Movies Dataset (v4115259)     |
| Total records    | ~2,590,928 titles                           |
| Time span        | 1920 – 2022 (102 years)                     |
| Key columns      | name, year, rating, certificate, duration, genre, votes, directors, stars |

---


## Key Analytical Questions

This notebook is structured to answer five core questions:

| # | Question |Section|
|---|----------|-------|
| 1 | How has global media production volume changed over 102 years? |3.1|
| 2 | Has audience engagement tracked production volume, or do the two diverge? |3.2|
| 3 | Which genres have grown, shrunk, or stayed dominant across each decade since the 1920s? |3.3|
| 4 | Which genres attract the most audience engagement — and does high volume always mean high engagement? |3.4|
| 5 | How has the balance of Kids, Family, and Adult-rated content shifted over time — in both volume and engagement? |3.5|

---



In [21]:
import pandas as pd 
import numpy as np 
import os
import re
import json
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go


df = pd.read_csv('/kaggle/input/imdb-movies-dataset/movies.csv',low_memory = False)


df['name'] = df['name'].str.strip()

del df['gross_income']
del df['description']

df.head()



,id,name,year,rating,certificate,duration,genre,votes,directors_id,directors_name,stars_id,stars_name
0,tt4710316,Best in Sex: 2015 AVN Awards,(2015 TV Special),4.0,TV-MA,94 min,"Adult, News",124.0,nm1624094,Gary Miller,"nm4766272,nm2670531,nm4920605,nm6284246","Farrah Laurel Abraham,Asa Akira,Anikka Albrite..."
1,tt1281857,Naughty Novelist,(2008 Video),3.8,Not Certified,88 min,Adult,174.0,nm0045256,John Bacchus,"nm0128986,nm1969196,nm0451160,nm6130462","Darian Caine,Jackie Stevens,A.J. Khan,Arrora"
2,tt2294954,2011 AVN Awards Show,(2011 TV Special),5.7,Not Certified,83 min,"Adult, News",39.0,"nm1624094,nm0754845","Gary Miller,Timothy E. Sabo","nm2200343,nm2670531,nm1267549,nm3585599","Aubrey Addams,Asa Akira,Monique Alexander,Rave..."
3,tt6843596,Best in Sex: 2017 AVN Awards,(2017 TV Special),4.9,TV-MA,87 min,"Adult, News",225.0,nm1624094,Gary Miller,"nm5221471,nm2670531,nm4920605,nm3038816","Amirah Adara,Asa Akira,Anikka Albrite,Britney ..."
4,tt3705604,AVN Awards 2014,(2014 TV Special),6.7,R,82 min,"Adult, News",101.0,nm1624094,Gary Miller,"nm2670531,nm4920605,nm6284246,nm3992720","Asa Akira,Anikka Albrite,August Ames,Jessie An..."


In [22]:
df.isnull().sum()
df.dropna(inplace = True)
df.isnull().sum()

id                0
name              0
year              0
rating            0
certificate       0
duration          0
genre             0
votes             0
directors_id      0
directors_name    0
stars_id          0
stars_name        0
dtype: int64

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2590928 entries, 0 to 2590931
Data columns (total 17 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    object 
 1   name                  object 
 2   year                  object 
 3   rating                float64
 4   certificate           object 
 5   duration              int64  
 6   genre                 object 
 7   votes                 int64  
 8   directors_id          object 
 9   directors_name        object 
 10  stars_id              object 
 11  stars_name            object 
 12  start_year            float64
 13  end_year              float64
 14  title_type            object 
 15  decade                float64
 16  certificate_category  object 
dtypes: float64(4), int64(2), object(11)
memory usage: 355.8+ MB


## 2. Data Cleaning

### 2.1. Cleaning Year Column

In [23]:
years_df = df['year'].str.extractall(r'(\b\d{4}\b)')[0].unstack(level='match')

df['start_year'] = years_df[0].fillna(0).astype(int)

df['end_year'] = years_df[1].fillna(years_df[0]).fillna(0).astype(int)

# classification for Series & Movie
years_str = df['year'].str.lower()

conditions = [
    years_str.str.contains('tv series|tv mini-series|tv miniseries', regex=True),
    years_str.str.contains('tv movie|tv special|video|short', regex=True),
]
choices = ['Series', 'Movie']

df['title_type'] = np.select(conditions, choices, default='Unknown')


rest = df['title_type'] == 'Unknown'
df.loc[rest, 'title_type'] = np.where(
    df.loc[rest, 'start_year'] != df.loc[rest, 'end_year'],
    'Series',
    'Movie'
)


### 2.2. Cleaning Duration Column

In [25]:
df['duration'] = df['duration'].str.replace(' min', '')
df['duration'] = df['duration'].str.replace(',' , '')

df['duration'] = df['duration'].astype(int)

### 2.3. Cleaning Genre Column

In [26]:
all_genre = []
df['genre'] = df['genre'].str.replace(' ', '')
for i in tqdm(df['genre'].str.split(',')):
    all_genre += i
len(all_genre)

clean_genre = list(set(all_genre))

  0%|          | 0/2590928 [00:00<?, ?it/s]

In [27]:
df['genre'].str.replace(' ', '').str.split(',').explode().value_counts()

genre
Drama          516398
Comedy         452896
Short          394409
Animation      388765
Adventure      348034
Game-Show      324968
Action         312765
Crime          271023
Family         243254
Sport          241713
Reality-TV     237513
Music          230634
Mystery        174145
Horror         164013
Documentary    155474
Thriller       151519
History        148420
Fantasy        147531
Romance         99986
Sci-Fi          98745
Biography       93549
Musical         84862
Talk-Show       82860
News            48059
War             32128
Western         29232
Film-Noir         771
Adult              50
Name: count, dtype: int64

### 2.4. Cleaning Votes Column

In [28]:
df['votes'] = df['votes'].str.replace(',', '').astype(float).astype(int)

### 2.5. Cleaning Directors_id & name Column

In [14]:
df['directors_id'] = df['directors_id'].str.replace('Anonymous','nm0000000').str.replace('/name/', '').str.replace('/','')
df['directors_name'] = df['directors_name'].str.replace('nm0000000','Anonymous')

director_ids = []
director_names = []

for i , j in df[['directors_id', 'directors_name']].values:
    
    if len(i.split(',')) != len(j.split(',')):
        director_ids.append(np.nan)
        director_names.append(np.nan)
    else:
        director_ids.append(i)
        director_names.append(j)

df['director_ids'] = director_ids
df['director_names'] = director_names


del df['directors_id']
del df['directors_name']


#### 2.5.1. Mapping directords id with their names

In [15]:
import json

directors_dict = dict(zip(director_ids, director_names))

with open('directors.json', 'w') as f:
    json.dump(directors_dict, f)
    

### 2.6. Cleaning Starts_id & name Column

In [16]:
df['stars_id'] = df['stars_id'].str.replace('Anonymous', 'nm0000000')
df['stars_name'] = df['stars_name'].str.replace('nm0000000', 'Anonymous')

star_ids = []
star_names = []

for i , j in df[['stars_id', 'stars_name']].values:
    if len(i.split(',')) != len(j.split(',')):
        star_ids.append(np.nan)
        star_names.append(np.nan)
    else:
        star_ids.append(i)
        star_names.append(j)
df['star_ids'] = star_ids
df['star_names'] = star_names

del df['stars_id']
del df['stars_name']
    

#### 2.6.1. Mapping Stars_ids and their names

In [17]:
stars_dict = dict(zip(star_ids, star_names))

with open('stars.json', 'w') as f1:
    json.dump(stars_dict, f1)

In [19]:
df.to_csv('IMDB Clean Data.csv')

## Data Cleaning Summary

Before analysis, the following issues were identified and treated:

- **Year column** — stored as raw strings e.g. `(2018 TV Series–2022)`. Cleaned via  
  vectorised regex extraction into `start_year` and `end_year` integer columns.

- **Movie vs. Series classification** — determined by keyword extraction from the raw  
  year string (`TV Series`, `TV Mini-Series`, `TV Movie` etc.) with year-range fallback.  
  Pure `start_year == end_year` logic was rejected as it misclassifies single-season series.

- **Genre column** Exploded for all genre-level analyses.
  28 distinct genres identified.

- **Directors & Stars** — stored as comma-separated ID strings. Exploded and mapped  
  to name strings via a built JSON lookup. `Anonymous` entries normalised.

---


## 3. Data Analysis

### 3.1. Trend of  Movies & Series released over the years

In [31]:
# DATA AGGREGATION
total_trend = df[(df['start_year'] >= 1920) & (df['start_year'] <= 2022)]['start_year'].value_counts().sort_index()

df_movies = df[df['title_type'] == 'Movie']
movies_trend = df_movies[(df_movies['start_year'] >= 1920) & (df_movies['start_year'] <= 2022)]['start_year'].value_counts().sort_index()

df_series = df[df['title_type'] != 'Movie']
series_trend = df_series[(df_series['start_year'] >= 1920) & (df_series['start_year'] <= 2022)]['start_year'].value_counts().sort_index()

# IMDB COLOR PALETTE 
imdb_bg = '#121212'            
imdb_yellow = '#F5C518'        
imdb_text_primary = '#F5C518'   
imdb_text_secondary = '#AAAAAA' 
imdb_grid = '#2A2A2A'
imdb_text_legend = '#FFFFFF'
highlight_yellow = 'rgba(245,197,24,0.15)'      
imdb_orange = '#FF8C00'
imdb_gradient_scale = [[0, imdb_yellow], [1, imdb_orange]]
cinematic_colors = [
    imdb_yellow, '#5799EF', '#FF8C00', '#E74C3C', 
    '#1ABC9C', '#9B59B6', '#F39C12', '#3498DB', '#E67E22', '#8E44AD']

fig = go.Figure()

# Adding 'All Media' Legend
fig.add_trace(go.Scatter(
    x=total_trend.index, 
    y=total_trend.values,
    name='All Media',
    mode='lines',
    line=dict(color='#555555', width=2, dash='dash'),
    hovertemplate='Total Titles: %{y}<extra></extra>'
))

# Adding 'Movies' Legend
fig.add_trace(go.Scatter(
    x=movies_trend.index, 
    y=movies_trend.values,
    name='Movies',
    mode='lines',
    line=dict(color=imdb_yellow, width=2),
    hovertemplate='Movies: %{y}<extra></extra>'
))

# Adding 'Series' Legend
fig.add_trace(go.Scatter(
    x=series_trend.index, 
    y=series_trend.values,
    name='Series',
    mode='lines',
    line=dict(color='#5799EF', width=2),
    hovertemplate='Series: %{y}<extra></extra>'
))

# Adding the Annotations
insights_html = (
    "<b>KEY ANALYST INSIGHTS:</b><br><br>"
    "• <b>The Streaming Boom:</b> from ~10k/year in 1980 to >110k at the 2019 peak.<br>"
    "• <b>The COVID Cliff:</b> over 60% drop between 2019 and 2022.<br>"
    "• <b>Movie Dominance:</b> films account for roughly 80% of all media volume.<br>"
    "• <b>Peak Series:</b>quadrupled since 1990."
)
fig.add_annotation(
    text=insights_html,
    xref="x", yref="y",
    x=1922, y=110000,          
    showarrow=False,            
    align="left",
    font=dict(
        family='Helvetica Neue, Arial, sans-serif',
        size=14,
        color=imdb_text_secondary
    ),          
    bordercolor=imdb_yellow,    
    borderwidth=1,
    borderpad=15,               
    xanchor="left",
    yanchor="top",
    bgcolor=highlight_yellow
)

# GRAPH CUSTOMIZATION 
fig.update_layout(
    
    title=dict(
        text='<b>IMDB | Global Media Production Trends (1920 - 2022)</b>',
        font=dict(color=imdb_text_primary, size=25, family='Arial, Helvetica, DejaVu Sans'),
        x=0.5
    ),
    paper_bgcolor=imdb_bg,
    plot_bgcolor=imdb_bg,
    font=dict(family='Arial, Helvetica, DejaVu Sans', color=imdb_text_secondary),
    
   
    hovermode='x unified',
    
    legend=dict(
        orientation="h", 
        yanchor="bottom", y=1.02, 
        xanchor="right", x=1,
        font=dict(color=imdb_text_legend, size=11)
    ),
    
    # X-Axis with INTERACTIVE YEAR SLIDER
    xaxis=dict(
        title=dict(text='Release Year', font=dict(size=12)),
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=10),
        rangeslider=dict(
            visible=True,
            bgcolor='#222222',
            thickness=0.1
        ),
        type='linear'
    ),
    
   
    yaxis=dict(
        title=dict(text='Number of Titles Released', font=dict(size=12)),
        gridcolor=imdb_grid,
        gridwidth=0.7,
        showline=False,
        zeroline=False,
        tickfont=dict(size=10)
    ),
    
    margin=dict(l=60, r=40, t=80, b=40)
)


fig.show()

### 3.2. Trend of Movies & Series based on User Engagement over the Years

In [32]:
# DATA AGGREGATION

total_votes = df[(df['start_year'] >= 1920) & (df['start_year'] <= 2022)].groupby('start_year')['votes'].sum()

movies_votes = df_movies[(df_movies['start_year'] >= 1920) & (df_movies['start_year'] <= 2022)].groupby('start_year')['votes'].sum()

series_votes = df_series[(df_series['start_year'] >= 1920) & (df_series['start_year'] <= 2022)].groupby('start_year')['votes'].sum()

fig = go.Figure()

# Adding 'All Media' Legend
fig.add_trace(go.Scatter(
    x=total_votes.index, 
    y=total_votes.values,
    name='All Media',
    mode='lines',
    line=dict(color='#555555', width=2, dash='dash'),
    hovertemplate='Total Votes: %{y:,.0f}<extra></extra>' # UX upgrade: comma formatting
))

# Adding 'Movies' Legend
fig.add_trace(go.Scatter(
    x=movies_votes.index, 
    y=movies_votes.values,
    name='Movies',
    mode='lines',
    line=dict(color=imdb_yellow, width=2),
    hovertemplate='Movie Votes: %{y:,.0f}<extra></extra>'
))

# Adding 'Series' Legend
fig.add_trace(go.Scatter(
    x=series_votes.index, 
    y=series_votes.values,
    name='Series',
    mode='lines',
    line=dict(color='#5799EF', width=2),
    hovertemplate='Series Votes: %{y:,.0f}<extra></extra>'
))

# Adding the Annotations

insights_html = (
    "<b>KEY INSIGHTS:</b><br><br>"
    "• <b>The 2000s Explosion:</b> fan engagement completely took off once the internet became mainstream.<br>"
    "• <b>Movies > Series:</b> movies still get way more votes from audiences overall.<br>"
    "• <b>The TV Comeback:</b> In the last decade, TV shows have seen a huge spike in fan engagement."
)

fig.add_annotation(
    text=insights_html,
    xref="x", yref="paper", 
    x=1922, y=0.95,           
    showarrow=False,            
    align="left",
    font=dict(
        family='Helvetica Neue, Arial, sans-serif',
        size=14,
        color=imdb_text_secondary
    ),          
    bordercolor=imdb_yellow,    
    borderwidth=1,
    borderpad=15,               
    xanchor="left",
    yanchor="top",
    bgcolor=highlight_yellow
)

# GRAPH CUSTOMIZATION 
fig.update_layout(
    
    title=dict(
        text='<b>IMDB | Global Media Engagement Trends (1920 - 2022)</b>',
        font=dict(color=imdb_text_primary, size=25, family='Arial, Helvetica, DejaVu Sans'),
        x=0.5 
    ),
    paper_bgcolor=imdb_bg,
    plot_bgcolor=imdb_bg,
    font=dict(family='Arial, Helvetica, DejaVu Sans', color=imdb_text_secondary),
    
    hovermode='x unified',
    
    legend=dict(
        orientation="h", 
        yanchor="bottom", y=1.02, 
        xanchor="right", x=1,
        font=dict(color=imdb_text_legend, size=11)
    ),
    
    # X-Axis with INTERACTIVE YEAR SLIDER
    xaxis=dict(
        title=dict(text='Release Year', font=dict(size=12)),
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=10),
        rangeslider=dict(
            visible=True,
            bgcolor='#222222',
            thickness=0.1
        ),
        type='linear'
    ),
    
    yaxis=dict(
        title=dict(text='Total Audience Engagement', font=dict(size=12)),
        gridcolor=imdb_grid,
        gridwidth=0.7,
        showline=False,
        zeroline=False,
        tickfont=dict(size=10),
        tickformat='.2s'
    ),
    
    margin=dict(l=60, r=40, t=80, b=40)
)

fig.show()

### 3.3. Distribution of Different Genres on number of movies over the decades


In [40]:
# DATA AGGREGATION & CLEANING

df['decade'] = (df[(df['start_year'] >= 1920) & (df['start_year'] <= 2022)]['start_year']// 10) * 10

df_clean = df[['decade', 'genre']].dropna().copy()
df_clean['genre'] = df_clean['genre'].astype(str).str.replace(' ', '').str.split(',')
df_exploded = df_clean.explode('genre')

decade_genre_df = df_exploded.groupby(['decade', 'genre']).size().reset_index(name='frequency')
decades = sorted(decade_genre_df['decade'].unique())

def get_top_15_hbar(dec):
    temp_df = decade_genre_df[decade_genre_df['decade'] == dec]
    return temp_df.sort_values(by='frequency', ascending=True).tail(15)


# ANIMATION TIMING CONTROLS

frame_speed = 1500       
transition_speed = 1000 

# INITIALIZING THE INTERACTIVE FIGURE & FRAMES

fig = go.Figure()


initial_decade = decades[0]
initial_data = get_top_15_hbar(initial_decade)
initial_max_freq = initial_data['frequency'].max()

fig.add_trace(go.Bar(
    x=initial_data['frequency'], 
    y=initial_data['genre'],
    orientation='h', 
    hovertemplate='<b>%{y}</b><br>Frequency: %{x:,.0f}<extra></extra>',
    marker=dict(
        color=initial_data['frequency'], 
        colorscale=imdb_gradient_scale,   
        showscale=False                   
    )
))

# Create the Animation Frames
frames = []
for dec in decades:
    frame_data = get_top_15_hbar(dec)
    
    frame_max = frame_data['frequency'].max()
    if pd.isna(frame_max) or frame_max == 0:
        frame_max = 10 
        
    frames.append(go.Frame(
        data=[go.Bar(
            x=frame_data['frequency'], 
            y=frame_data['genre'],
            orientation='h',
            marker=dict(
                color=frame_data['frequency'], 
                colorscale=imdb_gradient_scale,
                showscale=False
            )
        )],
        name=str(int(dec)),
        
        
        layout=go.Layout(
            xaxis=dict(range=[0, frame_max * 1.25])
        )
    ))
fig.frames = frames

# Create the Slider Configuration
sliders = [{
    "active": 0,
    "yanchor": "top",
    "xanchor": "left",
    "currentvalue": {
        "font": {"size": 18, "color": imdb_text_primary},
        "prefix": "Decade: ",
        "visible": True,
        "xanchor": "right"
    },
    "transition": {"duration": transition_speed, "easing": "linear"},
    "pad": {"b": 10, "t": 50},
    "len": 1.0,
    "x": 0,
    "y": -0.15, 
    "font": {"color": imdb_text_secondary},
    "bgcolor": '#222222',
    "bordercolor": imdb_grid,
    "steps": [{
        "args": [
            [str(int(dec))],
            {"frame": {"duration": frame_speed, "redraw": False}, 
             "mode": "immediate", 
             "transition": {"duration": transition_speed, "easing": "linear"}}
        ],
        "label": f"{int(dec)}s",
        "method": "animate"
    } for dec in decades]
}]


# GRAPH CUSTOMIZATION

fig.update_layout(
    title=dict(
        text='<b>IMDB | Genre Evolution over the years (1920 - 2022)</b>',
        font=dict(color=imdb_text_primary, size=22, family='Arial, Helvetica, sans-serif'),
        x=0.5
    ),
    paper_bgcolor=imdb_bg,
    plot_bgcolor=imdb_bg,
    font=dict(family='Arial, Helvetica, sans-serif', color=imdb_text_secondary),
    
    xaxis=dict(
        title=dict(text='Number of Titles Released', font=dict(size=12)),
        gridcolor=imdb_grid,
        gridwidth=0.7,
        showline=False,
        zeroline=False,
        tickfont=dict(size=10),
        range=[0, initial_max_freq * 1.25] 
    ),
    
    yaxis=dict(
        title=dict(text='Genre', font=dict(size=12)),
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=11, color=imdb_text_primary) 
    ),
    
    sliders=sliders,
    margin=dict(l=120, r=40, t=80, b=120) 
)

# Play/Pause controls 
fig.update_layout(
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": frame_speed, "redraw": False}, "fromcurrent": True}],
             "label": "▶ Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "transition": {"duration": 0}}],
             "label": "⏸ Pause", "method": "animate"}
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0.05,
        "yanchor": "top",
        "font": {"color": imdb_text_primary},
        "bgcolor": '#2A2A2A',
        "bordercolor": imdb_yellow
    }]
)

fig.show()

### 3.4. Genre Based Distribution on user engagement over the decades

In [34]:
freq = []

for genre in tqdm(clean_genre):
   freq.append(int(df[df['genre'].str.contains(genre)]['votes'].sum()))

  0%|          | 0/28 [00:00<?, ?it/s]

In [35]:
# DATA AGGREGATION

df_genre = pd.DataFrame()
df_genre['genre'] = clean_genre
df_genre['frequency'] = freq
df_genre = df_genre.sort_values(by='frequency', ascending=False)

top_15_genres = df_genre.head(15)

fig = go.Figure()

# Plotting the Bar Chart 
fig.add_trace(go.Bar(
    x=top_15_genres['genre'],
    y=top_15_genres['frequency'],
    hovertemplate='<b>%{x}</b><br>Engagement: %{y:,.0f}<extra></extra>',
    marker=dict(
        color=top_15_genres['frequency'], 
        colorscale=imdb_gradient_scale,   
        showscale=False                   
    )
))

# GRAPH CUSTOMIZATION
fig.update_layout(
    title=dict(
        text='<b>IMDB | Top 15 Genres by User Engagement (1920-2022)</b>',
        font=dict(color=imdb_text_primary, size=25, family='Arial, Helvetica, DejaVu Sans'),
        x=0.5
    ),
    paper_bgcolor=imdb_bg,
    plot_bgcolor=imdb_bg,
    font=dict(family='Arial, Helvetica, DejaVu Sans', color=imdb_text_secondary),
    
    # X-Axis Customization
    xaxis=dict(
        title=dict(text='Genre', font=dict(size=12)),
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=11),
        tickangle=45 
    ),
    
    # Y-Axis Customization
    yaxis=dict(
        title=dict(text='Frequency (Total Votes)', font=dict(size=12)),
        gridcolor=imdb_grid,
        gridwidth=0.7,
        showline=False,
        zeroline=False,
        tickfont=dict(size=10)
    ),
    
    margin=dict(l=60, r=40, t=80, b=80) 
)

fig.show()

### 3.5. Evolution Of Volume of Kids vs Family vs Adult Content over the years

In [36]:
kids      = ['TV-Y', 'TV-Y7', 'TV-Y7-FV', 'TV-G', 'GA', 'G','E', 'EC', 'CE', 'K-A', '6+']
family    = ['PG', 'TV-PG', 'PG-12', 'PG-13', 'GP', 'Approved', 'Passed', 'Open', 'TV-14', '12','TV-13', 'E10+', 'T', 'M/PG']
adult     = ['R', 'R-12', 'R-15', 'R-18', 'NC-17', 'X','AO', 'MA-13', 'MA-17', 'TV-MA', 'M','18', 'Banned', '(Banned)']
not_rated = ['Not Rated','Not Certified', 'Unrated']

In [37]:
mapping = {}

mapping.update({k: 'Kids' for k in kids})
mapping.update({k: 'Family' for k in family})
mapping.update({k: 'Adult' for k in adult})
mapping.update({k: 'Not_Rated' for k in not_rated})

df['certificate_category'] = df['certificate'].map(mapping)
df_categ = df[df['certificate_category'] != 'Not_Rated']

In [38]:
# DATA AGGREGATION

df_clean = df_categ[(df_categ['start_year'] >= 1950) & (df_categ['start_year'] <= 2022)]

cert_trend = df_clean.groupby(['start_year', 'certificate_category']).size().unstack(fill_value=0)
ideal_order = ['Kids', 'Family', 'Adult']

valid_order = [col for col in ideal_order if col in cert_trend.columns]

cert_trend = cert_trend[valid_order]

# INITIALIZING THE INTERACTIVE FIGURE

fig = go.Figure()


for i, category in enumerate(cert_trend.columns):
    fig.add_trace(go.Bar(
        x=cert_trend.index,      
        y=cert_trend[category],  
        name=str(category),
        
        marker_color=cinematic_colors[i % len(cinematic_colors)], 
        
        hovertemplate='<b>Year: %{x}</b><br>Rating: ' + str(category) + '<br>Count: %{y:,.0f}<extra></extra>'
    ))


# GRAPH CUSTOMIZATION

fig.update_layout(

    width = 1300,
    height = 700,
    barmode='stack', 
    
    title=dict(
        text='<b>IMDB | Certificate Distribution over the years (1950-2022)</b>',
        font=dict(color=imdb_text_primary, size=22, family='Arial, Helvetica, sans-serif'),
        x=0.5
    ),
    paper_bgcolor=imdb_bg,
    plot_bgcolor=imdb_bg,
    font=dict(family='Arial, Helvetica, sans-serif', color=imdb_text_secondary),
    
    
    hovermode='x unified',
    
    
    legend=dict(
        orientation="h", 
        yanchor="top", y=1.07,
        xanchor="center", x=0.5,
        font=dict(color=imdb_text_legend, size=11)
    ),
    
    # X-Axis with the interactive Range Slider
    xaxis=dict(
        title=dict(text='Release Year', font=dict(size=12)),
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=10),
        rangeslider=dict(
            visible=True,
            bgcolor='#222222',
            thickness=0.1
        ),
        type='linear'
    ),
    
    # Y-Axis Styling
    yaxis=dict(
        title=dict(text='Number of Titles Released', font=dict(size=12)),
        gridcolor=imdb_grid,
        gridwidth=0.7,
        showline=False,
        zeroline=False,
        tickfont=dict(size=10)
    ),
    
   
    margin=dict(l=80, r=60, t=90, b=80) 
)

fig.show()

In [39]:
# DATA AGGREGATION
df_clean = df_categ[(df_categ['start_year'] >= 1950) & (df_categ['start_year'] <= 2022)]

cert_votes_trend = df_clean.groupby(['start_year', 'certificate_category'])['votes'].sum().unstack(fill_value=0)
ideal_order = ['Kids', 'Family', 'Adult']

valid_order = [col for col in ideal_order if col in cert_votes_trend.columns]

cert_votes_trend = cert_votes_trend[valid_order]



# INITIALIZING THE INTERACTIVE FIGURE

fig = go.Figure()

for i, category in enumerate(cert_votes_trend.columns):
    fig.add_trace(go.Bar(
        x=cert_votes_trend.index,      
        y=cert_votes_trend[category],  # Now plots Total Votes instead of Count
        name=str(category),
        marker_color=cinematic_colors[i % len(cinematic_colors)], 
        
        # UX Upgrade: %{y:,.0f} adds commas to the massive vote counts (e.g., 1,500,000)
        hovertemplate='<b>Year: %{x}</b><br>Rating: ' + str(category) + '<br>Total Votes: %{y:,.0f}<extra></extra>'
    ))


#  GRAPH CUSTOMIZATION
fig.update_layout(
    width=1250,  
    height=750,   
    barmode='stack', 
    
    title=dict(
        text='<b>IMDB | Certificate Engagement Over the years (1950-2022)</b>',
        font=dict(color=imdb_text_primary, size=22, family='Arial, Helvetica, sans-serif'),
        x=0.5
    ),
    paper_bgcolor=imdb_bg,
    plot_bgcolor=imdb_bg,
    font=dict(family='Arial, Helvetica, sans-serif', color=imdb_text_secondary),
    
    hovermode='x unified',
    
   
    legend=dict(
        orientation="h", 
        yanchor="bottom", y=1, 
        xanchor="center", x=0.5,
        font=dict(color=imdb_text_legend, size=11)
    ),
    
    # X-Axis with Range Slider
    xaxis=dict(
        title=dict(text='Release Year', font=dict(size=12)),
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=10),
        rangeslider=dict(
            visible=True,
            bgcolor='#222222',
            thickness=0.1
        ),
        type='linear'
    ),
    
    # Y-Axis Styling 
    yaxis=dict(
        title=dict(text='Engagement on Different Categories', font=dict(size=12)),
        gridcolor=imdb_grid,
        gridwidth=0.7,
        showline=False,
        zeroline=False,
        tickfont=dict(size=10)
    ),
    
    margin=dict(l=80, r=40, t=80, b=120) 
)

fig.show()

---

## 4. Conclusions & Business Insights

### What the data tells us

- **Production volume peaked in 2019 and collapsed during COVID:**  
  Global titles released grew from ~10,000/year in 1980 to a peak of **114,229 in 2019**,  
  then fell over 60% to ~30,000 by 2022.

- **Movies account for ~83% of all titles but the Series gap is closing fast:**  
  In 1990 series represented roughly 35% of total output. By 2019 that share had risen to  
  over 42%, and series vote engagement quadrupled in the same window.

- **Drama is the universal genre glue:**  
  Drama appears in **516,398 titles** — more than any other genre.

- **Fan engagement is an internet-era phenomenon, not a content-era one:**  
  Total votes across all titles were negligible before 1995. The inflection point is  
  **1999–2002**, directly coinciding with IMDB's mainstream adoption post-Amazon acquisition  
  (1998). 

- **Adult-rated content engagement was virtually zero before 1995 — then exploded with the internet:**  
  The inflection in Adult content begins sharply at **1995–2010**,  
  directly tracking mainstream internet adoption.

  

---